In [0]:
# Day 4 - Databricks Notebook Data Engineering
 ## london_transport_day4_databricks_notebook
from pyspark.sql import functions as F
from pyspark.sql.types import *


# DBTITLE 1,
RAW_BASE_PATH = "/Volumes/london_transport_catalog/london_transport_raw_data/london_transport_exercise_volume/raw/"

### Reading the data

In [0]:
stations_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/stations.csv")
)
zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/zones.csv")
)
lines_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/lines.csv")
)
vehicle_types_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/vehicle_types.csv")
)
operators_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/operators.csv")
)
fares_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/fares.csv")
)
boroughs_df = (
    spark.read 
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_BASE_PATH}/boroughs.csv")
)

journeys_df = spark.read.option("multiline", True).json(f"{RAW_BASE_PATH}/journeys.json")
disruptions_df = spark.read.option("multiline", True).json(f"{RAW_BASE_PATH}/disruptions.json")


schedules_df = (
    spark.read
    .format("xml")
    .option("rowTag", "schedule")
    .load(f"{RAW_BASE_PATH}/schedules.xml")
)

### Cleaning data for later puposes

In [0]:
stations_clean_df = (
    stations_df
    .withColumn("station_id", F.trim(F.col("station_id")))
    .withColumn("station_name", F.initcap(F.trim(F.col("station_name"))))
    .withColumn("borough_id", F.trim(F.col("borough_id")))
    .dropDuplicates(["station_id"])
    .filter(F.col("station_id").isNotNull())
)

lines_clean_df = (
    lines_df
    .withColumn("line_id", F.trim(F.col("line_id")))
    .withColumn("line_name", F.initcap(F.trim(F.col("line_name"))))
    .withColumn("operator_id", F.trim(F.col("operator_id")))
    .withColumn("vehicle_type_id", F.trim(F.col("vehicle_type_id")))
    .dropDuplicates(["line_id"])
    .filter(F.col("line_id").isNotNull())
)

boroughs_clean_df = (
    boroughs_df
    .withColumn("borough_id", F.trim(F.col("borough_id")))
    .withColumn("borough_name", F.initcap(F.trim(F.col("borough_name"))))
    .dropDuplicates(["borough_id"])
    .filter(F.col("borough_id").isNotNull())
)

zones_clean_df = (
    zones_df
    .withColumn("zone_id", F.trim(F.col("zone_id")))
    .dropDuplicates(["zone_id"])
    .filter(F.col("zone_id").isNotNull())
)


journeys_clean_df = (
    journeys_df
    .filter(F.trim(F.col("delay_minutes")).rlike("^[0-9]+$"))
    .filter(F.col("passenger_count") != 'many')
    .withColumn("journey_id", F.trim(F.col("journey_id")))
    .withColumn("line_id", F.trim(F.col("line_id")))
    .withColumn("station_id", F.trim(F.col("station_id")))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("delay_minutes", F.col("delay_minutes").cast("int"))
    .dropDuplicates(["journey_id"])
    .filter(F.col("journey_id").isNotNull())
    .filter(F.col("passenger_count").isNotNull())
    .filter(F.col("passenger_count") >= 0)
    .filter(F.col("delay_minutes").isNotNull())
    .filter(F.col("delay_minutes") >= 0)
)

### Creating intermediate level df for later composing the curated data

In [0]:
transport_activity_df = (
    journeys_clean_df.alias("j")
    .join(lines_clean_df.alias("l"), F.col("j.line_id") == F.col("l.line_id"), "left")
    .join(stations_clean_df.alias("s"), F.col("j.station_id") == F.col("s.station_id"), "left")
    .join(boroughs_clean_df.alias("b"), F.col("s.borough_id") == F.col("b.borough_id"), "left")
    .select(
        F.col("j.journey_id"),
        F.col("j.line_id"),
        F.col("l.line_name"),
        F.col("j.station_id"),
        F.col("s.station_name"),
        F.col("b.borough_name"),
        F.col("j.passenger_count"),
        F.col("j.delay_minutes")
    )
)


line_operations_df = (
    lines_clean_df.alias("l")
    .join(operators_df.alias("o"), F.col("l.operator_id") == F.col("o.operator_id"), "left")
    .join(vehicle_types_df.alias("v"), F.col("l.vehicle_type_id") == F.col("v.vehicle_type_id"), "left")
    .select(
        F.col("l.line_id"),
        F.col("l.line_name"),
        F.col("o.operator_name"),
        F.col("v.vehicle_type_name")
    )
)

disruptions_by_line_df = (
    disruptions_df.alias("d")
    .join(lines_clean_df.alias("l"), F.col("d.line_id") == F.col("l.line_id"), "left")
    .select(
        F.col("d.disruption_id"),
        F.col("d.line_id"),
        F.col("l.line_name"),
        F.col("d.disruption_type"),
        F.col("l.service_status")
    )
)

fare_summary_df = (
    fares_df.alias("f")
    .join(zones_clean_df.alias("z"), F.col("f.zone_id") == F.col("z.zone_id"), "left")
    .select(
        F.col("f.fare_id"),
        F.col("f.zone_id"),
        F.col("z.zone_name"),
        F.col("f.fare_amount")
    )
)

### Creating the curated tables

In [0]:
top_stations_df = (
    transport_activity_df
    .groupBy("station_id", "station_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)


line_delay_report_df = (
    transport_activity_df
    .groupBy("line_id", "line_name")
    .agg(F.avg("delay_minutes").alias("average_delay_minutes"))
    .orderBy(F.desc("average_delay_minutes"))
)


borough_passenger_report_df = (
    transport_activity_df
    .groupBy("borough_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)


disruption_summary_df = (
    disruptions_by_line_df
    .groupBy("disruption_type")
    .agg(F.count("*").alias("disruption_count"))
    .orderBy(F.desc("disruption_count"))
)



line_passenger_report_df = (
    transport_activity_df
    .groupBy("line_id", "line_name")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy(F.desc("total_passengers"))
)

### Writing the curated df to files

In [0]:
transport_activity_df.write.mode("overwrite").saveAsTable("transport_activity_report")

In [0]:
top_stations_df.write.mode("overwrite").saveAsTable("top_stations")
line_delay_report_df.write.mode("overwrite").saveAsTable("line_delay_report")
borough_passenger_report_df.write.mode("overwrite").saveAsTable("borough_passenger_report")
disruption_summary_df.write.mode("overwrite").saveAsTable("disruption_summary")
line_passenger_report_df.write.mode("overwrite").saveAsTable("line_passenger_report")

In [0]:
%sql
SELECT * 
FROM top_stations
LIMIT 10;

# What was the main goal of Day 4 in this project?
Your answer:
## In your answer, explain:

### why the project moved into Databricks
Because we must learn Databricks.
Apart from that because so we can also forget having our computers at work busy calculating things before sending the data to someone. 
We will avoid things like:
- "it's not working" - "hu, but it runs on my computer"
- It's also available for anyone in charge of their analisys

### what stayed the same from the earlier project days
Basically everything except the part of setting up the whole vscode IDE thing

### what was new on Day 4
Databricks environment

# Checkpoint Question 2


Your answer:
## What did you learn about the Databricks environment?
Is like Jupyter Notebook but with everything connected and without running it in my computer consuming resources. Amazing.
### In your answer, explain in simple words:

#### what a workspace is

A folder that contains your whole project:
 - Python codes
 - Databases
 - Volumes (like files and stuff)

#### what a notebook is
A small python environment where you don't have to care about installing dependencies for ETL.
#### what compute means

The processing part of running a programm. The whole weight lifting thing.

#### why these ideas matter when doing data engineering work

Because engineers like 3 basic things:
- Efficiency, specially in costs. The less compute, the cheaper
- Scalability, meaning the day you are supposed to make your code process bigger amounts of data.
- Junk Food

# Checkpoint Question 3
Your answer:
## How did you use Spark in Databricks to work with the London Transport data?
Is extremely easy, you don't have to install it or import it. You just load the data to the dataframe from the location in the Databricks volumes.

## describe the main engineering workflow you followed.

Read the data -> Clean the data -> Create the base dfs that we will use for the curated reports -> Create the curated reports -> Store them

# Checkpoint Question 4
Your answer: 
## Why was it important to clean the data before building the final outputs?

There is a lot of noise and is also required to transform the formats in case some compute is necessary. If you have a '32' and you need to add 3, is impossible due to the different formats.

## In your answer, explain what kinds of raw data problems existed and how those problems could affect joins, reports, or business analysis if they were not fixed.
Duplicated data, Columns saying that you have "many" passengers, missing values, people writing loNdon instead of London...
Another thing that Engineers like: STANDARISATION



# Checkpoint Question 5
Your answer
## What final business-ready outputs did you create, and how do they help the business?
The curated reports read to use from someone else

Explain what business question each one helps answer.

The delays on the lines, passenger counts by the boroughs, line passengers, most used stations...
